In [1]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# 0.Processing

In [2]:
expr_top10_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top10_activate_kwx.csv"   # ROI × gene
gene_coexp_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/02.co_expression/gene_co_expr_matrix_top10_kwx.csv"    # 原始共表达 64×64
stim_path       = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/eeg_left_kwx.csv"
out_dir         = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/04.virtual_knockout/"

# ========= 1) 读数据 =========
E = pd.read_csv(expr_top10_path, index_col=0)      # ROI × gene（含NaN）
S = pd.read_csv(stim_path, index_col=0)            # 行名=stimulated那一列
S = S.apply(pd.to_numeric, errors="coerce")        # 空白保持NaN，不填充

print("E shape:", E.shape)
print("S shape:", S.shape)

# ========= 2) 对齐 ROI =========
common = E.index.intersection(S.index).intersection(S.columns)
E = E.loc[common, :]
S = S.loc[common, common]

R = E.shape[0]
print("Aligned ROI count:", R)
print("Aligned E shape:", E.shape)
print("Aligned S shape:", S.shape)

E shape: (64, 1563)
S shape: (64, 64)
Aligned ROI count: 64
Aligned E shape: (64, 1563)
Aligned S shape: (64, 64)


# 1.Original Gene-EEG Correlation

In [3]:
# ========= 3) 基线：ROI×ROI gene co-expression + 与刺激矩阵相关 =========
# 关键修复：E 是 ROI×gene，所以要用 E.T.corr() 得到 ROI×ROI
coexp_full = E.T.corr(method="pearson")  # pairwise NaN handling
print("coexp_full shape:", coexp_full.shape)

iu = np.triu_indices(R, k=1)
g_full = coexp_full.values[iu]
s_full = S.values[iu]

mask_full = np.isfinite(g_full) & np.isfinite(s_full)
g0 = g_full[mask_full]
s0 = s_full[mask_full]

r_full, p_full = pearsonr(g0, s0)
print(f"[Baseline] r={r_full:.4f}, p={p_full:.3e}, N={len(g0)}")

coexp_full shape: (64, 64)
[Baseline] r=0.3319, p=7.229e-37, N=1381


# 2.Virtual Knockout

In [4]:
# ========= 4) 虚拟敲除（不删基因，只置NaN） =========
results = []
for gene in E.columns:
    E_ko = E.copy()
    E_ko[gene] = np.nan  # 虚拟KO：该基因整列变缺失

    coexp_ko = E_ko.T.corr(method="pearson")  # 仍然是 ROI×ROI
    g_ko = coexp_ko.values[iu]

    mask_ko = np.isfinite(g_ko) & np.isfinite(s_full)
    g1 = g_ko[mask_ko]
    s1 = s_full[mask_ko]

    if len(g1) < 30:
        continue

    r_ko, _ = pearsonr(g1, s1)
    gci = r_ko - r_full
    results.append((gene, r_ko, gci, len(g1)))

res = pd.DataFrame(results, columns=["gene", "r_after_KO", "GCI", "N_edges"])
res["r_full"] = r_full

print("KO done. genes tested:", res.shape[0])
print(res.head())

KO done. genes tested: 1563
       gene  r_after_KO       GCI  N_edges    r_full
0   HSPA12B    0.331889 -0.000015     1381  0.331904
1     IFT74    0.331839 -0.000065     1381  0.331904
2  METTL21A    0.331893 -0.000011     1381  0.331904
3     AURKA    0.331913  0.000009     1381  0.331904
4    STARD4    0.331885 -0.000019     1381  0.331904


# 3.Save

In [5]:
# ========= 5) 分组 + 保存 =========
genes_increase_coupling = res[res["GCI"] < 0].sort_values("GCI")                  # KO后下降 => 基因存在增强相关
genes_decrease_coupling = res[res["GCI"] > 0].sort_values("GCI", ascending=False) # KO后上升 => 基因存在削弱相关

genes_increase_coupling.to_csv(out_dir + "genes_increase_coupling_kwx.csv", index=False)
genes_decrease_coupling.to_csv(out_dir + "genes_decrease_coupling_kwx.csv", index=False)

gene_matrix = pd.DataFrame({
    "increase_coupling_genes": pd.Series(genes_increase_coupling["gene"].values),
    "decrease_coupling_genes": pd.Series(genes_decrease_coupling["gene"].values),
})

gene_matrix.to_csv(out_dir + "gene_groups_matrix_kwx.csv", index=False)


print("Saved:")
print(" - genes_increase_coupling_kwx.csv")
print(" - genes_decrease_coupling_kwx.csv")
print(" - gene_groups_matrix_kwx.csv")

Saved:
 - genes_increase_coupling_kwx.csv
 - genes_decrease_coupling_kwx.csv
 - gene_groups_matrix_kwx.csv
